In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import sunpy.visualization.colormaps as cm

import glob
from processing import *
from classical_estimates import classical_estimates
from fit_pv import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
deadpix_file = '/home/ulyanov/data/solo/phi/dead_pixels/phi-fdt-deadpix_20250915T140003_V202609091415C_0569150100.fits'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'
distortion_file = '/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz'

In [3]:
flat_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/temp/*cavity*.fits'))

print(flat_files)

['/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240330T050009_V202608262158C_0463300100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20240926T114503_V202608262134C_0469260100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241016T113003_V202608262111C_0470160100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241027T233003_V202608262048C_0470270100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20241202T123003_V202608262027C_0472020100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250119T210009_V202608262003C_0561190100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250310T080009_V202608261939C_0563100100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250915T140003_V202608261916C_0569150100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20250923T000503_V202608261853C_0569230100.fits', '/home/ulyanov/data/solo/phi/flat/temp/phi-fdt-flat_20260310T040003_V202608261828C_0663100

In [4]:
i = -2
cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

In [5]:
#folder = '/home/ulyanov/data/solo/phi/2026/'
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/'

folder_blos = '/home/ulyanov/data/solo/phi/2026/blos/'
folder_vlos = '/home/ulyanov/data/solo/phi/2026/vlos/'

files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040003_V202606241235C_0663100100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040405_V202606241235C_0663100125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T040805_V202606241336C_0663100150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T041205_V202606241336C_0663100175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T041605_V202606241434C_0663100200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T042005_V202606241434C_0663100225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-03-10/solo_L1_phi-fdt-ilam_20260310T042405_V202606241535C_0663100250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026

In [6]:
for file in files[:1]:

    data, header = process(file,
                           dark_file=dark_file,
                           deadpix_file=deadpix_file,
                           prefilter_file=prefilter_file,
                           #cavity_file=cavity_file,
                           flatfield_file=flat_file,
                           ghost_file=ghost_file,
                           distortion_file=distortion_file,
                           _realign=True,
                           _find_center=True,
                           _demodulate=True,
                           _correct_fringes=True,
                           _correct_crosstalk=True,
                           _calc_wavelengths=True,
                           _mask=True,
                           )

    Blos, Vlos = classical_estimates(data, header, lam=1e-3, niter=10, five_points=True)

    file_blos = generate_filename(file, prefix='blos', folder=folder_blos)
    file_vlos = generate_filename(file, prefix='vlos', folder=folder_vlos)
    #clone_fits(file, file_blos, Blos, header)
    #clone_fits(file, file_vlos, Vlos, header)

61.0 0.03


KeyboardInterrupt: 

In [12]:
plt.figure(figsize=(10,10))
plt.imshow(data[4,3], 'gray', vmin=-30, vmax=30)
plt.tight_layout()

In [19]:
plt.figure(figsize=(10,10))
plt.imshow(Blos, 'gray', vmin=-200, vmax=200)
plt.tight_layout()

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(Vlos, 'seismic', vmin=-3000, vmax=3000)
plt.tight_layout()

In [13]:
plt.figure(figsize=(10,10))
plt.imshow(Vlos - q, 'seismic', vmin=-30, vmax=30)
plt.tight_layout()

In [12]:
np.nanmedian(Vlos - q)

np.float64(-0.18898547850574232)

In [8]:
q = Vlos.copy()

In [31]:
q_V = 299792458 / 6173.341

np.nanmedian(Vlos - q) / q_V

np.float64(-0.00010141487693195124)